# RA DeepSpatial：Zhuang-ABCA-2

GPU0 / seed42 / subclass / 稀有≤1% / 8配置。先阅读 README 和 resources/README。正式实验在服务器运行，本Notebook不含预先填入的效果结果。

In [ ]:
from pathlib import Path
import os, sys, subprocess, json
PROJECT = Path.cwd().resolve()
if (PROJECT / "RA_DeepSpatial").is_dir():
    PROJECT = PROJECT / "RA_DeepSpatial"
elif PROJECT.name == "examples":
    PROJECT = PROJECT.parent
assert (PROJECT / "configs/abca2.json").is_file(), PROJECT
PYTHON = sys.executable  # 可改为已配置环境的 Python 绝对路径。
ENV = dict(os.environ, CUDA_VISIBLE_DEVICES="0", PYTHONUNBUFFERED="1")
CONFIG = "configs/abca2.json"
CACHE = "cache/abca2"
OUTPUT = "outputs/abca2_seed42"
def run_python(*arguments):
    command = [PYTHON, "-u", *map(str, arguments)]
    print("Running:", command, "cwd:", PROJECT, flush=True)
    with subprocess.Popen(command, cwd=PROJECT, env=ENV, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        status = process.wait()
    if status:
        raise RuntimeError(f"子进程退出码 {status}；实际报错见上方输出。命令：{command}")
def run_stage(stage, *extra):
    run_python("examples/run_abca2.py", "--config", CONFIG, "--cache", CACHE,
               "--output", OUTPUT, "--stage", stage, "--resume", *extra)
print("PROJECT:", PROJECT)
print("PYTHON:", PYTHON)
print(json.loads((PROJECT / CONFIG).read_text()))


## 1. 检查固定划分

训练58张；验证015/034/059；测试008/022/040/053/066。此步骤只读取切片深度与行数，不读隐藏XY/表达/标签。

In [ ]:
run_python("scripts/inspect_data.py", "--config", CONFIG)


## 2. 下载并检查Marker

把Supplementary Table 7放入resources；运行inspect查看真实表头。自动导入失败时按输出指定工作表和正向Marker列。不要用Table5的总基因表替代类型映射。

In [ ]:
RESOURCE = PROJECT / "resources/41586_2023_6812_MOESM8_ESM.xlsx"
MARKERS = PROJECT / "resources/markers.csv"
if not MARKERS.exists():
    assert RESOURCE.is_file(), "先按 resources/README.md 下载 Supplementary Table 7"
    run_python("scripts/import_markers.py", RESOURCE, "--inspect")
    # 查看上方表头后，必要时在此命令追加 --sheet / --header / --subclass-column / --marker-columns。
    run_python("scripts/import_markers.py", RESOURCE)
assert MARKERS.is_file()


## 3. 工程检查与训练缓存

测试使用临时合成数据；prepare只拟合58张训练数据。

In [ ]:
run_python("tests/test_pipeline.py")
run_stage("prepare")


## 4. 训练全部8种配置

base/control各100 epochs；残差配置共用control主干并追加100 epochs。中断后重新运行本单元即可。

In [ ]:
run_stage("train")


## 5. 验证

仅用训练端点生成，再读取3张验证表达/标签评价。若修改配置，必须使用新的CACHE/OUTPUT，重新训练和验证。

In [ ]:
run_stage("validate")
print((PROJECT / OUTPUT / "validation_summary.csv").read_text()[:12000])


## 6. 冻结协议并测试

锁定8个checkpoint及验证证据后才读5张测试真值。测试不用于选参。

In [ ]:
run_stage("freeze")
run_stage("test")
run_stage("summary", "--device", "cpu")


## 7. 可选：导出完整方法的虚拟体积

分平面写盘，默认200µm间隔。若要改变导出参数，请在首次实验前改配置；冻结后不改配置。

In [ ]:
# run_stage("export", "--arms", "full")
